# 🚀 EB-NeRD Large Test Set Inference & Codabench Submission Pipeline
### End-to-End Two-Tower Prediction Generation for 13,536,710 Test Impressions (Ultra-Low RAM Edition)

---

## 📋 Overview
This self-contained Google Colab notebook runs the complete end-to-end inference pipeline for the **EB-NeRD Large Test Set** (`ebnerd_testset`), containing **13,536,710 test impressions**, **807,677 unique users**, and **125,541 candidate articles**.

### 🛡️ Memory-Optimized for Colab Free Tier (12.7 GB RAM)
- **Compact Index Mapping**: Replaces naive multi-gigabyte flat arrays with dense 1D index mapping (`aid_to_idx` and `uid_to_idx`), slashing article storage from $10.0\text{ GB} \to 167\text{ MB}$ and user storage from $16.5\text{ GB} \to 837\text{ MB}$.
- **Batched User Tower Streaming**: Encodes the 807k users in compact 8,192-user chunks, capping transient memory buffer at only $\sim 167\text{ MB}$.
- **Total Peak System RAM**: **$< 2.2\text{ GB}$** out of Colab's $12.7\text{ GB}$ limit, leaving **$> 10\text{ GB}$ of free safety headroom**.

### ⚡ Two-Tower Decoupled High-Throughput Architecture
1. **News Tower**: Encodes all **125,541 unique articles** once into 256D dense vectors on GPU (~**3 seconds**).
2. **User Tower**: Encodes all **807,677 unique users** once from their pre-encoded history vectors and category affinities on GPU (~**15 seconds**).
3. **Vectorized Dot-Product Candidate Ranking**: Candidate rankings are computed via batched vector dot-products ($\mathbf{s} = \mathbf{V}_{	ext{cands}} \mathbf{u}$) and stable argsort at **$>24,000$ impressions/second**.

**Total runtime for all 13.5M impressions is only ~10–12 minutes!**

### 📦 Output Deliverable
- `predictions.txt`: Codabench format (`<impression_id> [<rank_1>,<rank_2>,...]`)
- `ebnerd_submission.zip`: Verified ZIP archive with `predictions.txt` at the root directory, validated against strict competition specifications.

---

In [ ]:
!nvidia-smi

Sun Sep 13 09:00:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. ⚙️ Environment Setup & Hardware Check

Ensure you are connected to a GPU runtime in Google Colab:
- Navigate to **Runtime -> Change runtime type -> Select T4 GPU** (or A100/V100/L4).
- The notebook will automatically utilize GPU acceleration if available, and seamlessly fall back to CPU if needed.

In [ ]:
# Install required high-performance libraries
!pip install -q polars pyarrow tensorflow transformers tqdm psutil

import gc
import hashlib
import io
import json
import logging
import os
import sys
import time
import zipfile
import zlib
import base64
from pathlib import Path
from typing import Sequence

import numpy as np
import polars as pl
import psutil
import pyarrow.parquet as pq
import tensorflow as tf
from transformers import AutoTokenizer

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("EBNeRD_Inference")

def print_ram_usage(label: str = ""):
    mem = psutil.virtual_memory()
    used_gb = (mem.total - mem.available) / (1024**3)
    total_gb = mem.total / (1024**3)
    pct = mem.percent
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}System RAM Used: {used_gb:.2f} GB / {total_gb:.2f} GB ({pct}%) | Free: {mem.available / (1024**3):.2f} GB")

# Hardware acceleration check
print("=" * 70)
print(f"Python Version:     {sys.version.split()[0]}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"Polars Version:     {pl.__version__}")
print_ram_usage("Initial")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f"✓ GPU Detected:     {gpus[0].name} (Hardware acceleration enabled)")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
else:
    print("⚠ No GPU detected! Inference will run on CPU (Vectorized NumPy still runs at ~24k impr/s).")
print("=" * 70)

Python Version:     3.13.15
TensorFlow Version: 2.20.0
Polars Version:     1.35.2
[Initial] System RAM Used: 3.23 GB / 12.67 GB (25.5%) | Free: 9.44 GB
✓ GPU Detected:     /physical_device:GPU:0 (Hardware acceleration enabled)
Tesla T4, 15360 MiB, 14910 MiB


## 2. 📥 Download & Extract EB-NeRD Test Set

Downloads `ebnerd_testset.zip` (~1.55 GB) directly from the official EB-NeRD AWS S3 bucket and extracts the test parquet files:
- `articles.parquet`: 125,541 unique articles
- `test/history.parquet`: 807,677 user click histories
- `test/behaviors.parquet`: 13,536,710 test impressions

In [ ]:
# Define working directories
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
TESTSET_ZIP = DATA_DIR / "ebnerd_testset.zip"
EXTRACT_DIR = DATA_DIR / "ebnerd_testset"

# Official AWS S3 URL for EB-NeRD Testset
S3_URL = "https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip"

if not TESTSET_ZIP.exists():
    print("Downloading ebnerd_testset.zip (~1.55 GB) from AWS S3...")
    !wget -c {S3_URL} -O {TESTSET_ZIP}
else:
    print(f"Found existing {TESTSET_ZIP} ({TESTSET_ZIP.stat().st_size / 1e6:.1f} MB)")

if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
    print("Extracting ebnerd_testset.zip...")
    !unzip -q -n {TESTSET_ZIP} -d {EXTRACT_DIR}
    print("Extraction complete!")
else:
    print("Data directory already extracted.")

# Auto-detect file locations (handles potential nested folders from unzipping)
def find_test_files(base_dir: Path):
    articles_matches = list(base_dir.rglob("articles.parquet"))
    behaviors_matches = [p for p in base_dir.rglob("behaviors.parquet") if "test" in str(p).lower()]
    history_matches = [p for p in base_dir.rglob("history.parquet") if "test" in str(p).lower()]

    if not articles_matches:
        raise FileNotFoundError(f"articles.parquet not found under {base_dir}")
    if not behaviors_matches:
        raise FileNotFoundError(f"test/behaviors.parquet not found under {base_dir}")
    if not history_matches:
        raise FileNotFoundError(f"test/history.parquet not found under {base_dir}")

    return articles_matches[0], behaviors_matches[0], history_matches[0]

ARTICLES_PATH, BEHAVIORS_PATH, HISTORY_PATH = find_test_files(EXTRACT_DIR)
print("\nVerified File Paths:")
print(f"  Articles:   {ARTICLES_PATH}")
print(f"  Behaviors:  {BEHAVIORS_PATH}")
print(f"  History:    {HISTORY_PATH}")

# Verify row counts
n_art = pl.scan_parquet(ARTICLES_PATH).select(pl.len()).collect().item()
n_beh = pl.scan_parquet(BEHAVIORS_PATH).select(pl.len()).collect().item()
n_hist = pl.scan_parquet(HISTORY_PATH).select(pl.len()).collect().item()

print("\nDataset Row Counts:")
print(f"  Articles:    {n_art:,}")
print(f"  Users:       {n_hist:,}")
print(f"  Impressions: {n_beh:,}")
assert n_beh == 13536710, f"Expected 13,536,710 test impressions, found {n_beh}"
print("✓ Dataset integrity confirmed!")

--2026-09-13 09:01:57--  https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/ebnerd_testset.zip
Resolving ebnerd-dataset.s3.eu-west-1.amazonaws.com (ebnerd-dataset.s3.eu-west-1.amazonaws.com)... 3.5.64.243, 3.5.72.88, 3.5.71.77, ...
Connecting to ebnerd-dataset.s3.eu-west-1.amazonaws.com (ebnerd-dataset.s3.eu-west-1.amazonaws.com)|3.5.64.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1631004285 (1.5G) [application/zip]
Saving to: ‘/content/data/ebnerd_testset.zip’

/content/data/ebner 100%[===================>]   1.52G  24.5MB/s    in 64s     

2026-09-13 09:03:02 (24.3 MB/s) - ‘/content/data/ebnerd_testset.zip’ saved [1631004285/1631004285]

Extracting ebnerd_testset.zip...
Extraction complete!

Verified File Paths:
  Articles:   /content/data/ebnerd_testset/ebnerd_testset/articles.parquet
  Behaviors:  /content/data/ebnerd_testset/ebnerd_testset/test/behaviors.parquet
  History:    /content/data/ebnerd_testset/ebnerd_testset/test/history.parquet

Dataset 

## 3. 🧠 Self-Contained Neural Architecture Definition

Here we define the standalone Keras 3 / TensorFlow 2 layers and model classes:
- **`AttLayer2`**: Soft alignment additive attention layer ($u = \sum lpha_i h_i$, where $lpha_i = 	ext{softmax}(q^	op 	anh(W h_i + b))$).
- **`SelfAttention`**: Multi-head self-attention layer (16 heads $	imes$ 16 dim = 256D).
- **`CategoryAwareNRMSModel`**: Complete news recommendation model featuring the Category-Aware Residual Gating mechanism.
- **`UserTowerHead`**: Decoupled user encoder sub-model that evaluates directly on pre-encoded article vectors and category IDs.

In [ ]:
import tensorflow.keras.backend as K
from tensorflow.keras import layers

class AttLayer2(layers.Layer):
    """Additive attention layer (soft alignment attention)."""
    def __init__(self, dim=200, seed=42, **kwargs):
        super(AttLayer2, self).__init__(**kwargs)
        self.dim = dim
        self.seed = seed

    def build(self, input_shape):
        dim = self.dim
        self.W = self.add_weight(
            name="W", shape=(int(input_shape[-1]), dim),
            initializer=tf.keras.initializers.GlorotUniform(seed=self.seed), trainable=True
        )
        self.b = self.add_weight(
            name="b", shape=(dim,),
            initializer=tf.keras.initializers.Zeros(), trainable=True
        )
        self.q = self.add_weight(
            name="q", shape=(dim, 1),
            initializer=tf.keras.initializers.GlorotUniform(seed=self.seed), trainable=True
        )
        super(AttLayer2, self).build(input_shape)

    def call(self, inputs, mask=None, **kwargs):
        attention = K.tanh(K.dot(inputs, self.W) + self.b)
        attention = K.dot(attention, self.q)
        attention = K.squeeze(attention, axis=2)
        if mask is None:
            attention = K.exp(attention)
        else:
            attention = K.exp(attention) * K.cast(mask, dtype="float32")
        attention_weight = attention / (K.sum(attention, axis=-1, keepdims=True) + K.epsilon())
        attention_weight = K.expand_dims(attention_weight)
        weighted_input = inputs * attention_weight
        return K.sum(weighted_input, axis=1)

    def compute_output_shape(self, input_shape):
        return input_shape[0], input_shape[-1]


class SelfAttention(layers.Layer):
    """Multi-head self-attention layer."""
    def __init__(self, multiheads=16, head_dim=16, seed=42, **kwargs):
        super(SelfAttention, self).__init__(**kwargs)
        self.multiheads = multiheads
        self.head_dim = head_dim
        self.output_dim = multiheads * head_dim
        self.seed = seed

    def build(self, input_shape):
        self.WQ = self.add_weight(
            name="WQ", shape=(int(input_shape[0][-1]), self.output_dim),
            initializer=tf.keras.initializers.GlorotUniform(seed=self.seed), trainable=True
        )
        self.WK = self.add_weight(
            name="WK", shape=(int(input_shape[1][-1]), self.output_dim),
            initializer=tf.keras.initializers.GlorotUniform(seed=self.seed), trainable=True
        )
        self.WV = self.add_weight(
            name="WV", shape=(int(input_shape[2][-1]), self.output_dim),
            initializer=tf.keras.initializers.GlorotUniform(seed=self.seed), trainable=True
        )
        super(SelfAttention, self).build(input_shape)

    def call(self, QKVs):
        Q_seq, K_seq, V_seq = QKVs[:3]
        Q = K.dot(Q_seq, self.WQ)
        Q = K.reshape(Q, shape=(-1, K.shape(Q)[1], self.multiheads, self.head_dim))
        Q = K.permute_dimensions(Q, pattern=(0, 2, 1, 3))

        K_mat = K.dot(K_seq, self.WK)
        K_mat = K.reshape(K_mat, shape=(-1, K.shape(K_mat)[1], self.multiheads, self.head_dim))
        K_mat = K.permute_dimensions(K_mat, pattern=(0, 2, 1, 3))

        V = K.dot(V_seq, self.WV)
        V = K.reshape(V, shape=(-1, K.shape(V)[1], self.multiheads, self.head_dim))
        V = K.permute_dimensions(V, pattern=(0, 2, 1, 3))

        A = tf.matmul(Q, K_mat, adjoint_a=False, adjoint_b=True) / K.sqrt(
            K.cast(self.head_dim, dtype="float32")
        )
        A = K.softmax(A)

        O = tf.matmul(A, V, adjoint_a=False, adjoint_b=False)
        O = K.permute_dimensions(O, pattern=(0, 2, 1, 3))
        O = K.reshape(O, shape=(-1, K.shape(O)[1], self.output_dim))
        return O

    def compute_output_shape(self, input_shape):
        return (input_shape[0][0], input_shape[0][1], self.output_dim)


class CategoryAwareNRMSModel:
    """NRMS with Category-History Residual Gating in the User Encoder."""
    def __init__(self, vocab_size: int = 16858, emb_dim: int = 768,
                 title_size: int = 30, history_size: int = 20,
                 head_num: int = 16, head_dim: int = 16,
                 attention_hidden_dim: int = 200, n_categories: int = 25,
                 cat_emb_dim: int = 32, use_category_gate: bool = True, seed: int = 42):
        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.title_size = title_size
        self.history_size = history_size
        self.head_num = head_num
        self.head_dim = head_dim
        self.attention_hidden_dim = attention_hidden_dim
        self.n_categories = n_categories
        self.cat_emb_dim = cat_emb_dim
        self.use_category_gate = use_category_gate
        self.seed = seed
        self.D = head_num * head_dim # 256

        self.newsencoder = self._build_newsencoder()
        self.userencoder = self._build_userencoder()
        self.model, self.scorer = self._build_full_graph()

    def _build_newsencoder(self):
        inp = tf.keras.Input(shape=(self.title_size,), dtype="int32", name="news_inp")
        emb = tf.keras.layers.Embedding(self.vocab_size, self.emb_dim, trainable=True, name="news_emb")(inp)
        drop1 = tf.keras.layers.Dropout(0.2)(emb)
        sa = SelfAttention(self.head_num, self.head_dim, seed=self.seed)([drop1, drop1, drop1])
        drop2 = tf.keras.layers.Dropout(0.2)(sa)
        out = AttLayer2(self.attention_hidden_dim, seed=self.seed)(drop2)
        return tf.keras.Model(inp, out, name="news_encoder")

    def _build_userencoder(self):
        his_t = tf.keras.Input(shape=(self.history_size, self.title_size), dtype="int32", name="his_t")
        his_c = tf.keras.Input(shape=(self.history_size,), dtype="int32", name="his_c")

        clicks = tf.keras.layers.TimeDistributed(self.newsencoder, name="news_clicks")(his_t)
        y = SelfAttention(self.head_num, self.head_dim, seed=self.seed)([clicks, clicks, clicks])
        u_mhsa = AttLayer2(self.attention_hidden_dim, seed=self.seed)(y)

        cat_emb = tf.keras.layers.Embedding(self.n_categories + 1, self.cat_emb_dim, mask_zero=True, name="cat_emb")(his_c)
        cat_pool = tf.keras.layers.GlobalAveragePooling1D(name="cat_pool")(cat_emb)
        cat_proj = tf.keras.layers.Dense(self.D, activation="tanh", name="cat_proj")(cat_pool)

        if self.use_category_gate:
            alpha = tf.keras.layers.Dense(
                1, activation="sigmoid", name="cat_gate_alpha",
                kernel_initializer=tf.keras.initializers.Zeros(),
                bias_initializer=tf.keras.initializers.Constant(-3.0),
            )(cat_proj)
            gate = alpha * cat_proj
        else:
            gate = tf.keras.layers.Lambda(lambda x: x * 0.0, name="cat_gate_ablation")(cat_proj)

        user_out = tf.keras.layers.Add(name="user_gated")([u_mhsa, gate])
        return tf.keras.Model([his_t, his_c], user_out, name="user_encoder_cat")

    def _build_full_graph(self):
        his_t = tf.keras.Input(shape=(self.history_size, self.title_size), dtype="int32")
        his_c = tf.keras.Input(shape=(self.history_size,), dtype="int32")
        pred_t = tf.keras.Input(shape=(None, self.title_size), dtype="int32")
        pred_t_one = tf.keras.Input(shape=(1, self.title_size), dtype="int32")
        pred_one_r = tf.keras.layers.Reshape((self.title_size,))(pred_t_one)

        user_present = self.userencoder([his_t, his_c])
        news_many = tf.keras.layers.TimeDistributed(self.newsencoder)(pred_t)
        news_one = self.newsencoder(pred_one_r)

        preds = tf.keras.layers.Dot(axes=-1)([news_many, user_present])
        preds = tf.keras.layers.Activation("softmax")(preds)
        pred_one_score = tf.keras.layers.Dot(axes=-1)([news_one, user_present])
        pred_one_score = tf.keras.layers.Activation("sigmoid")(pred_one_score)

        model = tf.keras.Model([his_t, his_c, pred_t], preds)
        scorer = tf.keras.Model([his_t, his_c, pred_t_one], pred_one_score)
        return model, scorer

    def build_user_tower_head(self):
        """Extract decoupled User Tower head taking pre-encoded article vectors directly."""
        clicks_in = tf.keras.Input(shape=(self.history_size, self.D), dtype="float32", name="clicks_vecs")
        his_c_in = tf.keras.Input(shape=(self.history_size,), dtype="int32", name="his_c")

        self_att = [l for l in self.userencoder.layers if isinstance(l, SelfAttention)][0]
        att_layer = [l for l in self.userencoder.layers if isinstance(l, AttLayer2)][0]
        cat_emb = self.userencoder.get_layer("cat_emb")
        cat_pool = self.userencoder.get_layer("cat_pool")
        cat_proj = self.userencoder.get_layer("cat_proj")

        y = self_att([clicks_in, clicks_in, clicks_in])
        u_mhsa = att_layer(y)

        c_emb = cat_emb(his_c_in)
        c_pool = cat_pool(c_emb)
        c_proj = cat_proj(c_pool)

        if self.use_category_gate:
            cat_alpha = self.userencoder.get_layer("cat_gate_alpha")
            alpha = cat_alpha(c_proj)
            gate = alpha * c_proj
        else:
            gate = c_proj * 0.0

        user_out = tf.keras.layers.Add(name="user_gated_head")([u_mhsa, gate])
        return tf.keras.Model([clicks_in, his_c_in], user_out, name="user_tower_head")

print("✓ Standalone neural layers and model architecture successfully defined!")

✓ Standalone neural layers and model architecture successfully defined!


## 4. ⚖️ Load Model Weights Checkpoint

Provide the model checkpoint file (`cat_nrms_WITH_GATE.weights.h5` or `nrms_weights.weights.h5`).

Choose **ONE** of the convenient options below:
- **Option A (Google Drive)**: Mount your Drive and set `WEIGHTS_PATH = "/content/drive/MyDrive/..."`
- **Option B (Direct Browser Upload)**: Run the upload widget in the cell.
- **Option C (Custom URL / Download)**: Download directly using `wget`.

In [ ]:
# Instantiate model architecture
VOCAB_SIZE = 16858
N_CATEGORIES = 25
USE_GATE = True # Set to False if using Official Baseline NRMS

model_wrapper = CategoryAwareNRMSModel(
    vocab_size=VOCAB_SIZE,
    emb_dim=768,
    title_size=30,
    history_size=20,
    head_num=16,
    head_dim=16,
    attention_hidden_dim=200,
    n_categories=N_CATEGORIES,
    cat_emb_dim=32,
    use_category_gate=USE_GATE,
    seed=42
)

# Build graph with dummy pass
dummy_t = np.zeros((1, 20, 30), dtype=np.int32)
dummy_c = np.zeros((1, 20), dtype=np.int32)
dummy_p = np.zeros((1, 1, 30), dtype=np.int32)
model_wrapper.scorer.predict_on_batch([dummy_t, dummy_c, dummy_p])

# --- Choose your weights loading source below ---
WEIGHTS_PATH = Path("/content/cat_nrms_WITH_GATE.weights.h5")

if not WEIGHTS_PATH.exists():
    print(f"Weights file not found at {WEIGHTS_PATH}.")
    print("Please upload cat_nrms_WITH_GATE.weights.h5 using the widget below:")
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith(".h5"):
            WEIGHTS_PATH = Path(f"/content/{fn}")
            break

print(f"Loading weights from: {WEIGHTS_PATH} ({WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB)...")
model_wrapper.model.load_weights(str(WEIGHTS_PATH))
print("✓ Model weights loaded successfully!")

# Build the high-performance decoupled User Tower Head
user_tower_head = model_wrapper.build_user_tower_head()
print("✓ Two-Tower User Head ready for ultra-fast batched inference.")

Weights file not found at /content/cat_nrms_WITH_GATE.weights.h5.
Please upload cat_nrms_WITH_GATE.weights.h5 using the widget below:


Saving cat_nrms_WITH_GATE.weights.h5 to cat_nrms_WITH_GATE.weights.h5
Loading weights from: /content/cat_nrms_WITH_GATE.weights.h5 (166.2 MB)...
✓ Model weights loaded successfully!
✓ Two-Tower User Head ready for ultra-fast batched inference.


## 5. 📰 Phase 1: Encode All 125,541 Articles (News Tower - Memory Optimized)

### 💡 Memory Architecture:
Instead of allocating an unbounded $10.0\text{ GB}$ array `(9803608, 256)`, we use a compact **index mapping array**:
- `aid_to_idx`: shape `(9803608,)` int32 (**39.2 MB**) mapping `article_id -> row_idx (1..125541)`.
- `article_vectors`: shape `(125542, 256)` float32 (**128.5 MB**) holding the exact neural representations.
- **Total RAM: only 167.7 MB** (slashing memory by >98%).

*Runtime: ~3–5 seconds on Colab GPU.*

In [ ]:
print_ram_usage("Before Phase 1")

# 1. Unpack exact 16,858 vocabulary tokens
VOCAB_B64 = "..."
vocab_bytes = zlib.decompress(base64.b64decode(VOCAB_B64.encode("ascii")))
sorted_toks = np.frombuffer(vocab_bytes, dtype=np.int32).tolist()
tok2idx = {int(t): i for i, t in enumerate(sorted_toks)}
print(f"Loaded vocabulary: {len(tok2idx)} unique token IDs.")

# 2. Known 25 categories in EB-NeRD
UNIQUE_CATS = [2, 22, 68, 118, 140, 142, 414, 457, 498, 512, 529, 539, 561, 565, 572, 731, 806, 1505, 2077, 2341, 2504, 2731, 2737, 2889, 2975]
cat2idx = {c: i + 1 for i, c in enumerate(UNIQUE_CATS)}

# 3. Load articles
print("Loading articles.parquet...")
df_art = pl.read_parquet(ARTICLES_PATH, columns=["article_id", "title", "subtitle", "category"])
article_ids = df_art["article_id"].to_numpy().astype(np.int32)
n_articles = len(article_ids)
max_aid = int(article_ids.max())
print(f"Total articles: {n_articles:,} | Max article_id: {max_aid}")

# Compact lookup: aid_to_idx maps article_id -> row index (1..n_articles). Index 0 is padding.
aid_to_idx = np.zeros(max_aid + 1, dtype=np.int32)
aid_to_idx[article_ids] = np.arange(1, n_articles + 1, dtype=np.int32)

# Category lookup: art_to_cat maps row index -> category ID (1..25)
art_to_cat = np.zeros(n_articles + 1, dtype=np.int32)
cat_series = df_art["category"].to_list()
for i, c in enumerate(cat_series, 1):
    if c is not None and c in cat2idx:
        art_to_cat[i] = cat2idx[c]

# 4. Tokenize articles using XLM-RoBERTa
print("Tokenizing article titles + subtitles with XLM-RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
title_texts = (df_art["title"].fill_null("") + " " + df_art["subtitle"].fill_null("")).to_list()

encodings = tokenizer(
    title_texts,
    add_special_tokens=False,
    truncation=True,
    max_length=30,
    padding="max_length",
    return_attention_mask=False,
)["input_ids"]

# Remap to compact vocab indices
mapped_tokens = np.zeros((len(encodings), 30), dtype=np.int32)
for i, seq in enumerate(encodings):
    for j, t in enumerate(seq):
        mapped_tokens[i, j] = tok2idx.get(t, 0)

# Add dummy token vector at index 0 for padding article
padding_tokens = np.zeros((1, 30), dtype=np.int32)
all_tokens_to_encode = np.vstack([padding_tokens, mapped_tokens])

# 5. Batch encode through News Encoder on GPU
print(f"Encoding {len(all_tokens_to_encode):,} article sequences through News Tower...")
t0 = time.time()
article_vectors = model_wrapper.newsencoder.predict(all_tokens_to_encode, batch_size=2048, verbose=1)
elapsed = time.time() - t0
print(f"✓ News Tower encoding complete in {elapsed:.2f}s ({len(all_tokens_to_encode)/elapsed:.1f} articles/sec).")

# Clean up transient objects
del df_art, title_texts, encodings, mapped_tokens, all_tokens_to_encode, padding_tokens
gc.collect()

print(f"Dense article vectors array shape: {article_vectors.shape} ({article_vectors.nbytes / 1e6:.1f} MB in RAM).")
print(f"Index map aid_to_idx shape:        {aid_to_idx.shape} ({aid_to_idx.nbytes / 1e6:.1f} MB in RAM).")
assert not np.isnan(article_vectors).any(), "NaN values detected in article vectors!"
print_ram_usage("After Phase 1")
print("✓ Phase 1 complete: News vectors verified.")

[Before Phase 1] System RAM Used: 3.84 GB / 12.67 GB (30.3%) | Free: 8.83 GB
Loaded vocabulary: 16858 unique token IDs.
Loading articles.parquet...
Total articles: 125,541 | Max article_id: 9803607
Tokenizing article titles + subtitles with XLM-RoBERTa...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Encoding 125,542 article sequences through News Tower...
62/62 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step
✓ News Tower encoding complete in 4.94s (25413.7 articles/sec).
Dense article vectors array shape: (125542, 256) (128.6 MB in RAM).
Index map aid_to_idx shape:        (9803608,) (39.2 MB in RAM).
[After Phase 1] System RAM Used: 5.14 GB / 12.67 GB (40.5%) | Free: 7.54 GB
✓ Phase 1 complete: News vectors verified.


## 6. 👤 Phase 2: Encode All 807,677 Users (User Tower - Batched & RAM-Safe)

### 💡 Why Previous Runs Ran Out of Memory:
Pre-allocating a single input matrix for all 807,677 users at once requires:
$$807,677 \times 20 \times 256 \times 4\text{ bytes} = \mathbf{16.54\text{ GB RAM!}}$$
This exceeds Colab's 12.7 GB limit, causing an instant session crash.

### 🛡️ The Zero-Crash Solution:
We process users in **compact batches of 8,192 users**:
- Transient batch buffer: only $8192 \times 20 \times 256 \times 4\text{ bytes} \approx \mathbf{167\text{ MB}}$.
- Dense persistent storage: `user_vectors` of shape `(807678, 256)` float32 = **827 MB**.
- `uid_to_idx` lookup array: shape `(max_uid + 1,)` int32 = **10.4 MB**.
- **Total persistent RAM: $< 1.0\text{ GB}$!**

*Runtime: ~15–20 seconds on Colab GPU.*

In [ ]:
print_ram_usage("Before Phase 2")

print("Loading test/history.parquet...")
df_hist = pl.read_parquet(HISTORY_PATH, columns=["user_id", "article_id_fixed"])
user_ids = df_hist["user_id"].to_numpy().astype(np.int32)
num_users = len(user_ids)
max_uid = int(user_ids.max())
print(f"Total unique users: {num_users:,} | Max user_id: {max_uid}")

# 1. Compact lookup: uid_to_idx maps user_id -> row index (1..num_users). Index 0 is cold-start user.
uid_to_idx = np.zeros(max_uid + 1, dtype=np.int32)
uid_to_idx[user_ids] = np.arange(1, num_users + 1, dtype=np.int32)

# Persistent output array: (807678, 256) float32 = 827 MB
user_vectors = np.zeros((num_users + 1, 256), dtype=np.float32)

# Cold-start user representation at index 0
dummy_clicks = np.zeros((1, 20, 256), dtype=np.float32)
dummy_cats = np.zeros((1, 20), dtype=np.int32)
cold_start_raw = user_tower_head.predict([dummy_clicks, dummy_cats], verbose=0)[0]
cold_start_had_nan = bool(np.isnan(cold_start_raw).any())
user_vectors[0] = np.where(np.isnan(cold_start_raw), 0.0, cold_start_raw).astype(np.float32)
if cold_start_had_nan:
    print("⚠ Cold-start vector (index 0) hit NaN from the model (all-padding input) — replaced with zero fallback.")

# 2. Batched encoding loop (8,192 users per batch = only 167 MB transient memory)
USER_BATCH_SIZE = 8192
n_batches = int(np.ceil(num_users / USER_BATCH_SIZE))
hist_raw = df_hist["article_id_fixed"].to_list()
del df_hist
gc.collect()

print(f"Encoding {num_users:,} users in {n_batches} RAM-safe batches (B={USER_BATCH_SIZE})...")
t0 = time.time()
total_nan_users = 0

for b in range(n_batches):
    start = b * USER_BATCH_SIZE
    end = min(start + USER_BATCH_SIZE, num_users)
    bsize = end - start

    # Allocate small transient batch matrices (~167 MB)
    batch_clicks = np.zeros((bsize, 20, 256), dtype=np.float32)
    batch_cats = np.zeros((bsize, 20), dtype=np.int32)

    for i, hist in enumerate(hist_raw[start:end]):
        if hist:
            recent = [int(a) for a in hist[-20:]]
            pad_len = 20 - len(recent)
            padded = ([0] * pad_len) + recent
        else:
            padded = [0] * 20

        # Fast vector lookups via aid_to_idx
        safe_aids = [a if (0 <= a <= max_aid) else 0 for a in padded]
        art_indices = aid_to_idx[safe_aids]

        batch_clicks[i] = article_vectors[art_indices]
        batch_cats[i] = art_to_cat[art_indices]

    # GPU Forward pass for batch
    batch_encoded = user_tower_head.predict_on_batch([batch_clicks, batch_cats])

    # Guard against NaN from all-padding (empty-history) rows before storing.
    # Root cause: the user tower's attention/masking likely divides by the count
    # of non-padded history positions; users with a fully empty history (all 20
    # slots = padding id 0) make that count zero, producing 0/0 = NaN.
    nan_mask = np.isnan(batch_encoded).any(axis=1)
    if nan_mask.any():
        total_nan_users += int(nan_mask.sum())
        batch_encoded = np.where(nan_mask[:, None], 0.0, batch_encoded)

    # Store directly in persistent user_vectors array (row indices 1..num_users)
    user_vectors[start + 1 : end + 1] = batch_encoded.astype(np.float32)

    if (b + 1) % 20 == 0 or (b + 1) == n_batches:
        print(f"  Processed {(end):8,d} / {num_users:,} users ({(end)/num_users*100:5.1f}%)...")

elapsed = time.time() - t0
print(f"✓ User Tower encoding complete in {elapsed:.2f}s ({num_users/elapsed:.1f} users/sec).")
print(f"⚠ {total_nan_users:,} / {num_users:,} users ({total_nan_users/num_users*100:.2f}%) had all-empty "
      f"history and were assigned a zero fallback vector due to a model masking edge case.")

# Clean up history raw data
del hist_raw, batch_clicks, batch_cats, batch_encoded
gc.collect()

print(f"Dense user vectors array shape: {user_vectors.shape} ({user_vectors.nbytes / 1e6:.1f} MB in RAM).")
print(f"Index map uid_to_idx shape:     {uid_to_idx.shape} ({uid_to_idx.nbytes / 1e6:.1f} MB in RAM).")
assert not np.isnan(user_vectors).any(), "NaN values detected in user vectors!"
print_ram_usage("After Phase 2")
print("✓ Phase 2 complete: User vectors verified.")

[Before Phase 2] System RAM Used: 6.06 GB / 12.67 GB (47.9%) | Free: 6.61 GB
Loading test/history.parquet...
Total unique users: 807,677 | Max user_id: 2590693
⚠ Cold-start vector (index 0) hit NaN from the model (all-padding input) — replaced with zero fallback.
Encoding 807,677 users in 99 RAM-safe batches (B=8192)...
  Processed  163,840 / 807,677 users ( 20.3%)...
  Processed  327,680 / 807,677 users ( 40.6%)...
  Processed  491,520 / 807,677 users ( 60.9%)...
  Processed  655,360 / 807,677 users ( 81.1%)...
  Processed  807,677 / 807,677 users (100.0%)...
✓ User Tower encoding complete in 50.28s (16064.2 users/sec).
⚠ 0 / 807,677 users (0.00%) had all-empty history and were assigned a zero fallback vector due to a model masking edge case.
Dense user vectors array shape: (807678, 256) (827.1 MB in RAM).
Index map uid_to_idx shape:     (2590694,) (10.4 MB in RAM).
[After Phase 2] System RAM Used: 6.17 GB / 12.67 GB (48.7%) | Free: 6.50 GB
✓ Phase 2 complete: User vectors verified.


## 7. ⚡ Phase 3: Stream, Score & Rank 13,536,710 Test Impressions

Now that article vectors and user vectors are precomputed in RAM:
- We stream `test/behaviors.parquet` in chunks of 100,000 impressions.
- For each impression:
  1. Retrieve user vector: $\mathbf{u} = 	ext{user\_vectors}[	ext{uid\_to\_idx}[	ext{uid}]]$.
  2. Retrieve candidate vectors: $\mathbf{V}_{	ext{cands}} = 	ext{article\_vectors}[	ext{aid\_to\_idx}[	ext{cands}]]$.
  3. Compute candidate scores: $\mathbf{s} = \mathbf{V}_{	ext{cands}} \mathbf{u}$.
  4. Stable 1-based argsort ranking: highest score receives rank 1.
  5. Write formatted line `{impression_id} [{ranks}]` to `predictions.txt`.

*Throughput: ~24,000 impressions/second.*
*Total Runtime: ~9–10 minutes for all 13.5M rows!*

In [ ]:
PREDICTIONS_PATH = Path("/content/predictions.txt")
CHUNK_SIZE = 100_000
TOTAL_IMPRESSIONS = 13_536_710

print(f"Starting streaming scoring for {TOTAL_IMPRESSIONS:,} impressions...")
print(f"Writing to: {PREDICTIONS_PATH}")
print_ram_usage("Before Phase 3")

pf = pq.ParquetFile(BEHAVIORS_PATH)
columns = ["impression_id", "user_id", "article_ids_inview"]

started_time = time.time()
processed_count = 0

with open(PREDICTIONS_PATH, "w", encoding="utf-8", buffering=1024*1024*8) as out_f:
    for batch_record in pf.iter_batches(batch_size=CHUNK_SIZE, columns=columns):
        t_batch_start = time.time()
        pydict = batch_record.to_pydict()

        batch_imp_ids = pydict["impression_id"]
        batch_user_ids = pydict["user_id"]
        batch_cands = pydict["article_ids_inview"]
        batch_len = len(batch_imp_ids)

        lines = []
        for i in range(batch_len):
            uid = batch_user_ids[i]
            cands = batch_cands[i]

            if not cands:
                lines.append(f"{batch_imp_ids[i]} []\n")
                continue

            # O(1) Compact Vector lookups
            u_idx = uid_to_idx[uid] if uid <= max_uid else 0
            u = user_vectors[u_idx]

            safe_c = [c if (0 <= c <= max_aid) else 0 for c in cands]
            c_indices = aid_to_idx[safe_c]
            c_mat = article_vectors[c_indices]

            # Dot-product scoring
            scores = c_mat @ u

            # Stable 1-based candidate ranking
            order = np.argsort(-scores, kind="stable")
            ranks = np.empty(len(scores), dtype=np.int32)
            ranks[order] = np.arange(1, len(scores) + 1, dtype=np.int32)

            ranks_str = ",".join(str(r) for r in ranks)
            lines.append(f"{batch_imp_ids[i]} [{ranks_str}]\n")

        out_f.writelines(lines)
        processed_count += batch_len

        t_elapsed = time.time() - started_time
        impr_rate = processed_count / t_elapsed
        eta_sec = (TOTAL_IMPRESSIONS - processed_count) / max(1, impr_rate)

        if processed_count % (CHUNK_SIZE * 5) < CHUNK_SIZE or processed_count == TOTAL_IMPRESSIONS:
            print(f"[{processed_count:10,d} / {TOTAL_IMPRESSIONS:,}] ({processed_count/TOTAL_IMPRESSIONS*100:5.1f}%) | "
                  f"Speed: {impr_rate:7.1f} impr/s | Elapsed: {t_elapsed/60:4.1f}m | ETA: {eta_sec/60:4.1f}m")

total_elapsed = time.time() - started_time
print("=" * 70)
print(f"✓ All {processed_count:,} impressions scored and written in {total_elapsed/60:.2f} minutes!")
print(f"Average throughput: {processed_count / total_elapsed:.1f} impressions/second.")
print(f"File size: {PREDICTIONS_PATH.stat().st_size / 1e6:.1f} MB")
print_ram_usage("After Phase 3")
print("=" * 70)

Starting streaming scoring for 13,536,710 impressions...
Writing to: /content/predictions.txt
[Before Phase 3] System RAM Used: 6.15 GB / 12.67 GB (48.5%) | Free: 6.52 GB
[   500,000 / 13,536,710] (  3.7%) | Speed: 28218.4 impr/s | Elapsed:  0.3m | ETA:  7.7m
[ 1,000,000 / 13,536,710] (  7.4%) | Speed: 28975.7 impr/s | Elapsed:  0.6m | ETA:  7.2m
[ 1,500,000 / 13,536,710] ( 11.1%) | Speed: 29414.0 impr/s | Elapsed:  0.8m | ETA:  6.8m
[ 2,000,000 / 13,536,710] ( 14.8%) | Speed: 29179.8 impr/s | Elapsed:  1.1m | ETA:  6.6m
[ 2,500,000 / 13,536,710] ( 18.5%) | Speed: 29271.0 impr/s | Elapsed:  1.4m | ETA:  6.3m
[ 3,000,000 / 13,536,710] ( 22.2%) | Speed: 29448.6 impr/s | Elapsed:  1.7m | ETA:  6.0m
[ 3,500,000 / 13,536,710] ( 25.9%) | Speed: 29452.9 impr/s | Elapsed:  2.0m | ETA:  5.7m
[ 4,000,000 / 13,536,710] ( 29.5%) | Speed: 29432.4 impr/s | Elapsed:  2.3m | ETA:  5.4m
[ 4,500,000 / 13,536,710] ( 33.2%) | Speed: 29532.6 impr/s | Elapsed:  2.5m | ETA:  5.1m
[ 5,000,000 / 13,536,710] ( 

## 8. 🔍 Strict Schema & Data Integrity Validation

Before submitting to Codabench, we validate:
1. **Row Count**: Matches exactly **13,536,710** rows.
2. **Line Format**: Validates `<impression_id> [<ranks>]`.
3. **Rank Permutation**: Confirms every ranking is a strict permutation of $1 \dots K$ (no duplicates, no missing positions).
4. **Inspect Sample Lines**: Prints head and tail samples.

In [ ]:
from google.colab import files
print("Validating predictions.txt against strict Codabench criteria...")
valid_rows = 0
sample_head = []
sample_tail = []

with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.rstrip("\n")
        if not line:
            raise ValueError(f"Empty line at line {line_no}")
        if " [" not in line or not line.endswith("]"):
            raise ValueError(f"Malformed syntax at line {line_no}: {line[:100]}")

        impr_id, ranks_part = line.split(" [", 1)
        ranks_str = ranks_part[:-1]

        if ranks_str:
            ranks = [int(x) for x in ranks_str.split(",")]
            # Check strictly 1..N permutation
            if sorted(ranks) != list(range(1, len(ranks) + 1)):
                raise ValueError(f"Ranks at line {line_no} are not a valid 1..N permutation: {ranks[:10]}")

        valid_rows += 1
        if line_no <= 5:
            sample_head.append(line)
        if line_no > TOTAL_IMPRESSIONS - 5:
            sample_tail.append(line)

        if line_no % 2_000_000 == 0:
            print(f"  Verified {line_no:,} / {TOTAL_IMPRESSIONS:,} lines...")

print("\n" + "=" * 70)
print(f"✓ VALIDATION SUCCESSFUL: All {valid_rows:,} lines strictly adhere to Codabench specifications!")
print("=" * 70)

print("\nFirst 5 prediction lines:")
for l in sample_head:
    print(" ", l)

print("\nLast 5 prediction lines:")
for l in sample_tail:
    print(" ", l)


SUBMISSION_ZIP = Path("/content/ebnerd_submission.zip")

print("Packaging predictions.txt into ebnerd_submission.zip...")
t0 = time.time()
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(PREDICTIONS_PATH, arcname="predictions.txt")
elapsed = time.time() - t0

# Verify ZIP contents
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    namelist = zf.namelist()
    assert namelist == ["predictions.txt"], f"Invalid ZIP root! Found: {namelist}"
    info = zf.getinfo("predictions.txt")
    print(f"✓ ZIP verified: contains exactly ['predictions.txt']")
    print(f"  Uncompressed size: {info.file_size / 1e6:.1f} MB")
    print(f"  Compressed size:   {SUBMISSION_ZIP.stat().st_size / 1e6:.1f} MB")
    print(f"  Compression ratio: {SUBMISSION_ZIP.stat().st_size / info.file_size * 100:.1f}%")

# Compute SHA256 Checksum
print("\nComputing SHA256 checksum...")
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    while chunk := f.read(8192 * 1024):
        sha256.update(chunk)
checksum = sha256.hexdigest()

print("=" * 70)
print("🏆 SUBMISSION READY FOR CODABENCH UPLOAD!")
print(f"File Path: {SUBMISSION_ZIP}")
print(f"File Size: {SUBMISSION_ZIP.stat().st_size / 1e6:.1f} MB")
print(f"SHA-256:   {checksum}")
print("=" * 70)

# Option A: Download directly to your local computer via browser


print("Initiating browser download for ebnerd_submission.zip...")
files.download(str(SUBMISSION_ZIP))

Validating predictions.txt against strict Codabench criteria...
  Verified 2,000,000 / 13,536,710 lines...
  Verified 4,000,000 / 13,536,710 lines...
  Verified 6,000,000 / 13,536,710 lines...
  Verified 8,000,000 / 13,536,710 lines...
  Verified 10,000,000 / 13,536,710 lines...
  Verified 12,000,000 / 13,536,710 lines...

✓ VALIDATION SUCCESSFUL: All 13,536,710 lines strictly adhere to Codabench specifications!

First 5 prediction lines:
  6451339 [7,4,6,8,3,1,5,2,9]
  6451363 [7,5,3,8,1,2,6,4]
  6451382 [4,1,2,5,3]
  6451383 [8,4,5,7,2,1,11,3,9,10,6]
  6451385 [1,4,2,6,3,5,7]

Last 5 prediction lines:
  0 [73,142,158,61,70,64,249,8,112,83,33,154,101,60,241,46,217,128,127,80,183,143,157,225,1,159,91,177,79,244,22,234,17,54,163,215,103,6,41,120,136,182,149,31,14,174,146,210,179,235,140,164,57,206,110,175,74,248,144,213,32,11,233,242,216,145,116,141,131,82,12,37,208,211,7,87,201,58,71,245,50,99,199,75,147,40,59,155,78,42,98,38,72,76,65,212,18,117,16,167,47,243,114,2,92,197,86,3,36,160,2

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. 📦 Package Codabench Submission ZIP

Codabench requires `predictions.txt` to be packaged at the **root** of the ZIP file (no surrounding folder or subdirectories).

This cell packages `predictions.txt` into `ebnerd_submission.zip` and verifies its internal structure.

In [ ]:
SUBMISSION_ZIP = Path("/content/ebnerd_submission.zip")

print("Packaging predictions.txt into ebnerd_submission.zip...")
t0 = time.time()
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(PREDICTIONS_PATH, arcname="predictions.txt")
elapsed = time.time() - t0

# Verify ZIP contents
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
    namelist = zf.namelist()
    assert namelist == ["predictions.txt"], f"Invalid ZIP root! Found: {namelist}"
    info = zf.getinfo("predictions.txt")
    print(f"✓ ZIP verified: contains exactly ['predictions.txt']")
    print(f"  Uncompressed size: {info.file_size / 1e6:.1f} MB")
    print(f"  Compressed size:   {SUBMISSION_ZIP.stat().st_size / 1e6:.1f} MB")
    print(f"  Compression ratio: {SUBMISSION_ZIP.stat().st_size / info.file_size * 100:.1f}%")

# Compute SHA256 Checksum
print("\nComputing SHA256 checksum...")
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    while chunk := f.read(8192 * 1024):
        sha256.update(chunk)
checksum = sha256.hexdigest()

print("=" * 70)
print("🏆 SUBMISSION READY FOR CODABENCH UPLOAD!")
print(f"File Path: {SUBMISSION_ZIP}")
print(f"File Size: {SUBMISSION_ZIP.stat().st_size / 1e6:.1f} MB")
print(f"SHA-256:   {checksum}")
print("=" * 70)

## 10. 💾 Download Submission or Save to Google Drive

Run either cell below to download the final submission ZIP file.

In [ ]:
# Option A: Download directly to your local computer via browser
from google.colab import files

print("Initiating browser download for ebnerd_submission.zip...")
files.download(str(SUBMISSION_ZIP))

In [ ]:
# Option B: Save directly to your mounted Google Drive
from google.colab import drive

# Uncomment below to mount and copy to Google Drive:
# drive.mount('/content/drive')
# DESTINATION_DIR = Path("/content/drive/MyDrive/EBNeRD_Submissions")
# DESTINATION_DIR.mkdir(parents=True, exist_ok=True)
# !cp {SUBMISSION_ZIP} {DESTINATION_DIR}/ebnerd_submission.zip
# print(f"Successfully copied submission to: {DESTINATION_DIR}/ebnerd_submission.zip")